In [1]:
from nemo.collections.asr.parts.submodules.ngram_lm import NGramGPULanguageModel
from nemo.collections.asr.parts.submodules.ctc_batched_beam_decoding import BatchedBeamCTCComputer
from inference_funcs import load_bit_phoneme_model, evaluate_model
from dataset import getDatasetLoaders
import torch.nn.functional as F
import numpy as np
import torch
from edit_distance import SequenceMatcher

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CHAR_VOCAB = [
    "<sp>",          # space token
    "!", ",", ".", "?", "'",   # punctuation (incl. apostrophe)
] + [chr(i) for i in range(ord('a'), ord('z') + 1)]  # 'a'..'z'

# Build mappings
_CHAR_TO_ID = {c: i for i, c in enumerate(CHAR_VOCAB)}
_ID_TO_CHAR = {i: c for c, i in _CHAR_TO_ID.items()}

# Convenience indices
SPACE_ID = _CHAR_TO_ID["<sp>"]

def charToId(c: str) -> int:
    """Map raw input char to ID, normalizing space and lowercase."""
    if c == " ":
        c = "<sp>"
    c = c.lower()
    return _CHAR_TO_ID[c]

def idToChar(i: int) -> str:
    return _ID_TO_CHAR[i]

In [3]:
language_model_path = "/data/lm/char_12gram_lm.nemo"
vocab_size = 32
lm = NGramGPULanguageModel.from_nemo(
    lm_path=language_model_path,
    vocab_size=vocab_size
)

[NeMo I 2025-09-17 01:59:32 save_restore_connector:282] Model NGramGPULanguageModel was successfully restored from /data/lm/char_12gram_lm.nemo.


In [4]:
device = 'cuda'
bit_phoneme_filepath = "/data/models/time_masked_transfomer_characters_phonemes_80ms_seed_0//"
model, args = load_bit_phoneme_model(bit_phoneme_filepath)
model = model.to(device)

In [5]:
data_file = '/data/neural_data/ptDecoder_ctc_both_char_phoneme'
trainLoaders, testLoaders, loadedData = getDatasetLoaders(
        data_file, 8, None, 
        False
    )

In [6]:
outputs, cer, per_day_cer, _, _ = evaluate_model(model, loadedData, args, partition='test', device='cuda')

CER DAY 0: 0.373233
CER2 DAY 0: 0.354586
CER DAY 1: 0.288823
CER2 DAY 1: 0.275862
CER DAY 2: 0.255659
CER2 DAY 2: 0.238019
CER DAY 3: 0.275168
CER2 DAY 3: 0.254317
CER DAY 4: 0.152788
CER2 DAY 4: 0.124664
CER DAY 5: 0.110843
CER2 DAY 5: 0.099812
CER DAY 6: 0.135662
CER2 DAY 6: 0.116367
CER DAY 7: 0.171516
CER2 DAY 7: 0.152406
CER DAY 8: 0.170501
CER2 DAY 8: 0.142629
CER DAY 9: 0.179943
CER2 DAY 9: 0.178333
CER DAY 10: 0.161169
CER2 DAY 10: 0.143482
CER DAY 11: 0.190287
CER2 DAY 11: 0.191176
CER DAY 12: 0.160377
CER2 DAY 12: 0.139405
CER DAY 13: 0.144928
CER2 DAY 13: 0.134768
CER DAY 14: 0.109924
CER2 DAY 14: 0.100803
CER DAY 15: 0.119756
CER2 DAY 15: 0.105596
CER DAY 16: 0.139493
CER2 DAY 16: 0.114286
CER DAY 17: 0.168908
CER2 DAY 17: 0.143139
CER DAY 18: 0.125465
CER2 DAY 18: 0.108647
CER DAY 19: 0.126476
CER2 DAY 19: 0.109576
CER DAY 20: 0.106255
CER2 DAY 20: 0.100503
CER DAY 21: 0.143902
CER2 DAY 21: 0.127460
CER DAY 22: 0.130709
CER2 DAY 22: 0.126969
CER DAY 23: 0.167993
CER2 DAY 2

In [9]:
import pickle
with open('/data/text/validation_sentences_ground_truth.pkl', 'rb') as f:
    val_ground_truth_all = pickle.load(f)

In [15]:
num_classes = 33
logits = np.zeros((len(outputs['logits']), max(outputs['logitLengths']), num_classes))
for idx, l in enumerate(outputs['logits']):
    l_length = outputs['logitLengths'][idx]
    logits[idx, :l_length, :] = l
    
T = 1
alpha = 1.5

logits_torch = torch.from_numpy(logits)
log_probs = F.log_softmax(logits_torch/T, dim=-1)
# Reorder so that blank is the last index
perm = torch.arange(1, log_probs.shape[-1], device=log_probs.device)  
perm = torch.cat([perm, torch.tensor([0], device=log_probs.device)]) 
log_probs = log_probs.index_select(-1, perm).contiguous()

log_probs_length = torch.from_numpy(np.array(outputs['logitLengths']))
decoder = BatchedBeamCTCComputer(blank_index=32, beam_size=200, 
                                 return_best_hypothesis=True, fusion_models=[lm], 
                                fusion_models_alpha=[alpha], beam_threshold=20)


In [16]:
idx = 50

log_probs_one_sample = torch.unsqueeze(log_probs[idx], dim=0)
log_probs_length_one_sample = torch.unsqueeze(log_probs_length[idx], dim=0)

In [17]:
transcripts = decoder.batched_beam_search_torch(log_probs_one_sample, log_probs_length_one_sample)
best_hyp_lm = transcripts.to_nbest_hyps_list()[0]
for i in range(len(best_hyp_lm.n_best_hypotheses)):
    decoded_lm = "".join(idToChar(idx) for idx in best_hyp_lm.n_best_hypotheses[i].y_sequence).replace('<sp>', ' ')
    print(f'{i}: {decoded_lm}')

In [18]:
best_hyp_lm = transcripts.to_nbest_hyps_list()

In [63]:
decoded_lm_arr = []
decoded_greedy_arr = []

for i in range(880):
    
    decoded_lm = "".join(idToChar(idx) for idx in best_hyp_lm[i].n_best_hypotheses[0].y_sequence).replace('<sp>', ' ').replace('.', '')
    decoded_lm.strip()
    decoded_lm_arr.append(decoded_lm)
    
    decoded_greedy = "".join(idToChar(idx) for idx in outputs['decodedSeqs'][i]-1).replace('<sp>', ' ').replace('.', '')
    decoded_greedy.strip()
    decoded_greedy_arr.append(decoded_greedy)

In [67]:
c = 0
for dl, dg, val in zip(decoded_lm_arr, decoded_greedy_arr, val_ground_truth_all):
    c += 1
    print(c, "Decoded LM: ", dl)
    print(c, "Decoded Greedy: ", dg)
    print(c, "Ground truth: ", val)

1 Decoded LM:  this rented
1 Decoded Greedy:  thecas reunted 
1 Ground truth:  theocracy reconsidered
2 Decoded LM:  each species have had it 
2 Decoded Greedy:  ach proshc sevel haund ligof 
2 Ground truth:  rich purchased several signed lithographs
3 Decoded LM:  so is we aned in an in 
3 Decoded Greedy:  so rolso we mained in anvash colsion 
3 Ground truth:  so rules we made in unabashed collusion
4 Decoded LM:  i am needed back is to be mpletely alient 
4 Decoded Greedy:  gro casome need back goves to be uempetly allont 
4 Ground truth:  lori's costume needed black gloves to be completely elegant
5 Decoded LM:  the truth ve got to come on or through file out 
5 Decoded Greedy:  the tuth vaay regot to hom one rasor thouth fi out 
5 Ground truth:  the tooth fairy forgot to come when roger's tooth fell out
6 Decoded LM:  that thing me with a bad reputations 
6 Decoded Greedy:  that shingng meppor with cos bu qraed reper i listion 
6 Ground truth:  that stinging vapor was caused by chl

In [65]:
from cer_wer import  _cer_and_wer
cer, wer, _ =  _cer_and_wer(decodedSentences=decoded_lm_arr, trueSentences=val_ground_truth_all)
print(cer, wer)
from cer_wer import  _cer_and_wer
cer, wer, _ =  _cer_and_wer(decodedSentences=decoded_greedy_arr, trueSentences=val_ground_truth_all)
print(cer, wer)

0.190339567822771 0.4066193853427896
0.2000726348284002 0.5039098017821422


In [48]:
idx = 400
log_probs_one_sample = torch.unsqueeze(log_probs[idx], dim=0)
log_probs_length_one_sample = torch.unsqueeze(log_probs_length[idx], dim=0)
print(log_probs_length_one_sample)
max_len = log_probs_length_one_sample[0]
log_probs_length_one_sample[0] = max_len
print(max_len)
transcripts = decoder.batched_beam_search_torch(log_probs_one_sample[0:1, 0:max_len, :], 
                                                log_probs_length_one_sample)
best_hyp_lm = transcripts.to_nbest_hyps_list()[0]
print(val_ground_truth_all[idx])

for i in range(len(best_hyp_lm.n_best_hypotheses)):
    decoded_lm = "".join(idToChar(idx) for idx in best_hyp_lm.n_best_hypotheses[i].y_sequence).replace('<sp>', ' ')
    print(f'{i}: {decoded_lm}')

tensor([61])
tensor(61)
friday afternoon at five thirty
